In [ ]:
import torch, torch.nn as nn, torch.optim as optim

corpus = ['The movie was fantastic', 'the movie was boring', 'the plot was amazing', 'i hated the plot']

In [6]:
words = sorted(set(' '.join(corpus).split()))
w2i = {'':0} | {w:i+1 for i,w in enumerate(words)}
i2w = {i:w for w,i in w2i.items()}
V, L = len(w2i), max(len(s.split()) for s in corpus)

In [7]:
X,y = [],[]
for s in corpus:
    t = [w2i[w] for w in s.split()]
    for i in range(1,len(t)):
        p = [0]*(L-i-1) + t[:i+1]
        X.append(p[:-1])
        y.append(p[-1])

X_t, y_t = torch.tensor(X), torch.tensor(y)

In [8]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.e = nn.Embedding(V,32)
        self.r = nn.RNN(32,32,batch_first=True)
        self.f = nn.Linear(32,V)
    def forward(self,x):
        _,h = self.r(self.e(x))
        return(self.f(h[-1]))

In [12]:
model = RNN()
opt = optim.Adam(model.parameters(),1e-3)
crit = nn.CrossEntropyLoss()

for e in range(500):
    opt.zero_grad()
    l = crit(model(X_t),y_t)
    l.backward()
    opt.step()
    if (e+1)%100==0:
        print(f'Loss: {l.item()}')


Loss: 0.4678993225097656
Loss: 0.20628120005130768
Loss: 0.1536850929260254
Loss: 0.1372368186712265
Loss: 0.12979717552661896


In [22]:
def predict(s):
    t = [w2i.get(w,0) for w in s.split()]
    p = [0]*(L-1-len(t)) + t
    with torch.no_grad():
        return i2w[model(torch.tensor([p])).argmax(1).item()]
    
print("The movie was -> ", predict("The movie was"))

The movie was ->  fantastic
